In [ ]:
%load_ext autoreload
%autoreload 2

# WW3 boundaries: real ERA5 2D wave spectra, end to end

Sets up a real, small CrocoDash case with WW3 as the **active** wave
component (`...SGLC_WW3_SESP` instead of the stub `SWAV`) and runs it
through `configure_forcings()`/`process_forcings()` using the **real** ERA5
2D wave spectra product (`raw_data_access/datasets/era5.py`) for WW3's own
OBC -- `ww3_obc_product_name`/`ww3_obc_function_name` are `WW3Configurator`
inputs, and `get_step_days`/`regrid_step_days` (also `WW3Configurator`
inputs) chunk the GET step by day so a slow/rate-limited CDS queue can be
resumed incrementally instead of one giant request. There's no separate
manual copy-and-slice step and no placeholder-then-real two-pass dance --
one `configure_forcings()` call, one `process_forcings()` call, same as
every other component (MOM6, CICE, ...).

Every real ERA5 grid point in each boundary's buffered window becomes its
own station (see `ww3.py`'s `_regrid_chunk_era5`/`_extract_all_stations`),
not collapsed to one point -- `ww3_bounc.nml`'s `INTERP` is chosen
accordingly (`2`/linear for multi-station, vs. `1`/nearest for the
placeholder `WW3` product's one-station-per-boundary case).

## Step 0: Paths and imports

Follows the "Fast CrocoDash Case Setup" pattern from the repo's `CLAUDE.md`
-- same small flat-bathymetry domain used there, just with WW3 swapped in.


In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

CESMROOT = Path("~/work/installs/full_regional_cesm").expanduser()
CASENAME = "ww3_boundaries_demo"
CASEROOT = (Path("~/croc_cases") / CASENAME).expanduser()
INPUTDIR = (Path("~/scratch/croc_input") / CASENAME).expanduser()

print(CESMROOT)
print(CASEROOT)
print(INPUTDIR)

## Step 1: Grid, topography, vertical grid

The exact domain from CLAUDE.md's fast-setup example (10x10 degrees at
1-degree resolution, flat 1000m bathymetry, uniform vgrid) -- this is a
wiring demo, not a science case, so there's no reason to use anything
fancier.

In [ ]:
grid = Grid(
    lenx=10.0,
    leny=10.0,
    resolution=1.0,
    xstart=-60.0,
    ystart=30.0,
    name=CASENAME,
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)
vgrid = VGrid.uniform(nk=10, depth=1000.0, name=CASENAME)

## Step 2: Create the case

Just `SWAV` -> `WW3_SESP` in the NYF compset to get the real coupled wave
model instead of the stub. (`SESP` is CESM's stub "external system
processing" component, unrelated to waves -- it's just what this compset
alias always appends after an active WAV component.)

**Before running the next cell**: `override=True` below deletes and recreates
`caseroot`/`inputdir` for `ww3_boundaries_demo` if they already exist. If a
previous run of this notebook already fetched some real ERA5 raw files here,
this wipes them too -- there's no resuming a fetch across a case recreation,
only within repeated `process_forcings()` calls against the same case (see
the next section).

In [ ]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3_SESP",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)
print("WW3 in compset:", case.ww3_in_compset)

## Step 3: Configure + process forcings

`configure_forcings` sets up MOM6's IC/OBC download (GLORYS via a generated
script on this system -- see the next cell) *and*, since WW3 is active, the
WW3 OBC configurator -- pointed at the **real** ERA5 2D wave spectra product
(`ww3_obc_product_name`/`ww3_obc_function_name`) with the GET step chunked
to one day per request (`get_step_days=1`). `process_forcings()` then runs
the driver workflow for everything active for this compset: MOM6 IC, MOM6
OBC, and WW3 OBC against real ERA5 data.

**This is slow and real**: each boundary's daily ERA5 request expands into
`24 hours * 24 directions * 30 frequencies` GRIB messages, and CDS's queue
delay (server congestion, not request size) can mean 30-60+ minutes per
boundary per day in practice -- a full 9-day, 4-boundary run can take
hours. `get_step_days=1` means a rerun of `process_forcings()` resumes
where it left off instead of starting over: the GET step skips any daily
file that's already on disk (see `utils.fetch_raw_chunk`).

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    ww3_obc_product_name="era5_wave_spectra",
    ww3_obc_function_name="get_era5_2d_spectra",
    get_step_days=1,
)

Restore any previously-fetched real ERA5 raw files from a backup (see
warning above) into the fresh case's raw-data directory. `get_step_days=1`'s
per-day idempotency check (`utils.fetch_raw_chunk`) skips any file that's
already there by exact filename, so this resumes real progress from an
earlier run instead of re-fetching everything from CDS. Safe to run even
with no backup present -- it just does nothing.

In [ ]:
import shutil

backup_dirs = sorted(
    Path("~/scratch/croc_input").expanduser().glob("ww3_boundaries_demo_raw_era5_backup_*")
)
raw_data_dir = INPUTDIR / "extract_forcings" / "ww3" / "raw_data"
raw_data_dir.mkdir(parents=True, exist_ok=True)

restored = 0
if backup_dirs:
    for f in backup_dirs[-1].glob("*.nc"):
        target = raw_data_dir / f.name
        if not target.exists():
            shutil.copy(f, target)
            restored += 1
    print(f"restored {restored} previously-fetched real ERA5 raw file(s) from {backup_dirs[-1]}")
else:
    print("no backup found -- starting fresh")

GLORYS access on this system returns a *script* rather than downloading
directly -- run it once (credentials already cached in
`~/.copernicusmarine/`), same as `dev/cice_obc_demo`'s notebook needed.
`process_forcings()`'s first attempt will also print out an *additional*
IC-specific download line appended to this same script -- run it again
after that to pick it up (both cells below handle that).

In [ ]:
import subprocess

raw_data_dir = INPUTDIR / "extract_forcings" / "raw_data"
subprocess.run(["bash", str(raw_data_dir / "get_glorys_data.sh")], check=True)

In [ ]:
case.process_forcings()

In [ ]:
# If the cell above stopped with FileNotFoundError for ic_unprocessed.nc,
# run the download script again (process_forcings appended the IC-specific
# line to it) and re-run the cell above.
subprocess.run(["bash", str(raw_data_dir / "get_glorys_data.sh")], check=True)

## Inspect the real multi-station ERA5 output

Every real ERA5 grid point in each boundary's buffered window is its own
station now -- hundreds across the domain's perimeter, not one synthetic
point per boundary -- each carrying its own genuine `E(f, theta)` spectrum,
left for `ww3_bounc`'s linear interpolation (`BOUND%INTERP = 2`) to blend
between.

In [ ]:
ocnice = INPUTDIR / "ocnice"

spec_names = (ocnice / "spec.list").read_text().splitlines()
print("total real stations across all 4 boundaries:", len(spec_names))
print(
    [l for l in (ocnice / "ww3_bounc.nml").read_text().splitlines() if "INTERP" in l][-1]
)

In [ ]:
import xarray as xr

lats, lons, maxes, nonzero = [], [], [], 0
for name in spec_names:
    ds = xr.open_dataset(ocnice / name, decode_times=False)
    lats.append(float(ds["latitude"].values[0]))
    lons.append(float(ds["longitude"].values[0]))
    maxes.append(float(ds["efth"].max()))
    if (ds["efth"].values != 0).any():
        nonzero += 1
    ds.close()

print(f"stations with real (nonzero) wave energy: {nonzero}/{len(spec_names)}")
print(f"latitude range:  {min(lats):.2f} to {max(lats):.2f}")
print(f"longitude range: {min(lons):.2f} to {max(lons):.2f}")
print(f"peak efth across all stations: {max(maxes):.3f} m^2 s / rad")

## Case is set up

`./case.build`/`./case.submit` are intentionally not run here -- this
notebook is about validating the OBC-generation pipeline, not running the
coupled model. That's a natural next step once this looks right.

In [ ]:
print("caseroot:", case.caseroot)
print("inputdir:", case.inputdir)